# Jacobik Conformer Ensemble Solver

This notebook demonstrates solving one of the predefined debug complexes from `interface/ligand_library.py` (`DEBUG_COMPLEX_CBTYCO`), resolving polyatomic ligands and binding modes from the library, and exporting ranked conformers.

In [ ]:
import importlib.util, sys
from pathlib import Path

repo = Path.cwd().resolve()
while repo != repo.parent and not (repo / "src").exists():
    repo = repo.parent
if importlib.util.find_spec("jacobik") is None and (repo / "src").is_dir():
    sys.path.insert(0, str(repo / "src"))
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import jacobik
from interface.ligand_library import (
    DEBUG_COMPLEX_CBTYCO,
    DEBUG_COMPLEX_OJIMIF,
    LIGANDS,
    ligand,
    ligands,
    list_ligands,
)

env_info = jacobik.check_environment()
print("Environment OK:", env_info)
print(f"Loaded ligand library ({len(LIGANDS)} polyatomic ligands)")


## 1. Inspect Constraints Database & Library Ligands

In [ ]:
db_path = jacobik.default_db_path()
print(f"Using constraints DB: {db_path}")
print("CBTYCO debug complex:", DEBUG_COMPLEX_CBTYCO.metal, DEBUG_COMPLEX_CBTYCO.geometry)
print("dmso modes:", list(LIGANDS["dmso"].modes.keys()))
print("Example O-bound dmso input:", ligand("dmso@kappa1_O"))


## 2. Solve `DEBUG_COMPLEX_CBTYCO` Directly

In [ ]:
result = jacobik.generate(
    *DEBUG_COMPLEX_CBTYCO,
    seed=11,
    n_conformers=3,
    device="cpu",
)
result.summary()


## 3. Build & Solve From Library Names / Binding Modes

In [ ]:
from interface.ligand_library import LigandInput


pt_mixed = jacobik.generate(
    metal="Eu",
    geometry="square_anti",
    charge=3,
    spin=1,
    ligands=[LigandInput("[N]1(CC(=[O])NC)CC[N](CC(=[O])NC)CC[N](CC(=[O])NC)CC[N](CC(=[O])NC)CC1", [[0], [8], [16], [24], [3], [11], [19], [27]])],
    seed=11,
    n_conformers=2,
    device="cpu",
    n_ik_steps=200
    lr=0.01
)
pt_mixed.summary()


Class  Rows  Best   d(nats)  Configuration
----------------------------------------------------------------
   C1     2     1       0.0  square_anti  sites #173  (1 reflected)
   C2     2     3     108.7  square_anti  sites #77  (1 reflected)

Rank       Score   d(nats)  Converged  Class
--------------------------------------------
   1     -1.4445       0.0         no     C1
   2     -1.4445       0.0         no     C1  reflected
   3     -3.2561     108.7         no     C2
   4     -3.2561     108.7         no     C2  reflected

0/4 converged, 2 classes


## 4. Export Top Conformers to XYZ

In [11]:
out_dir = repo / "out"
out_dir.mkdir(parents=True, exist_ok=True)
for c in pt_mixed.conformers:
    xyz_path = out_dir / f"eudtma_{c.rank}.xyz"
    c.write_xyz(xyz_path)
    print(f"Saved rank {c.rank} (score: {c.score:.4f}) to {xyz_path}")

assert len(result.conformers) > 0
print("Export complete!")


Saved rank 1 (score: -1.4445) to /mnt/e/Dev/Projects/jacobik_2026-10-02_full/out/eudtma_1.xyz
Saved rank 2 (score: -1.4445) to /mnt/e/Dev/Projects/jacobik_2026-10-02_full/out/eudtma_2.xyz
Saved rank 3 (score: -3.2561) to /mnt/e/Dev/Projects/jacobik_2026-10-02_full/out/eudtma_3.xyz
Saved rank 4 (score: -3.2561) to /mnt/e/Dev/Projects/jacobik_2026-10-02_full/out/eudtma_4.xyz


NameError: name 'result' is not defined